In [19]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set rendering default to browser view
pio.renderers.default = "browser"

# 1. Load the stored data packet vectors
# Replace filename with your latest lockin saved output file name
d = np.load("lockin_final_version_data_68.2873kHz_from_fg_1ms_time_constant.npz")

data      = d["data_main"].astype(np.uint64)
data_freq = d["data_freq"].astype(np.uint64)
fs        = d["fs"]

# Compute continuous time arrays
t = np.arange(len(data)) / fs
t_freq = np.arange(len(data_freq)) / fs

In [20]:
# ============================================================
# ===== FIXED DECODING - NEW LOCK-IN STREAM LAYOUT =====
# ============================================================
# Define your hardware's peak input voltage range 
# (Set to 1.0 if you just want normalized -1 to +1 range, or 1.0 for Red Pitaya RF inputs)
V_PEAK = 1.0 
INT16_FS = 32767.0

# --- MAIN STREAM UNPACKING [data_main] ---
# 1. Extract raw 16-bit signed integer components first
q_raw   = ((data >> 48) & 0xFFFF).astype(np.int16)
i_raw   = ((data >> 32) & 0xFFFF).astype(np.int16)
adc_raw = ((data >> 16) & 0xFFFF).astype(np.int16)
sin_raw = (data & 0xFFFF).astype(np.int16)

# 2. Convert to physical Volts (cast to float32 during math operation)
mix_q_lpf   = (q_raw / INT16_FS) * V_PEAK
mix_i_lpf   = (i_raw / INT16_FS) * V_PEAK
adc_aligned = (adc_raw / INT16_FS) * V_PEAK
sin_ref     = (sin_raw / INT16_FS) * V_PEAK

# Calculate Vector Amplitude (R) from the filtered I and Q quadratures
# Multiplied by 2 to account for the mixing loss factor inherent to analog multiplication
calculated_amplitude = 2.0 * np.sqrt(mix_i_lpf.astype(np.float32)**2 + mix_q_lpf.astype(np.float32)**2)

# --- FREQUENCY STREAM UNPACKING [data_freq] ---
period_avg   = ((data_freq >> 48) & 0xFFFF).astype(np.uint16)
ref_edge_pll = (data_freq >> 47) & 0x1
nco_edge_pll = (data_freq >> 46) & 0x1
ftw_selected = data_freq & ((1 << 46) - 1)

# Convert 46-bit FTW back to physical Hz metrics
FCW = 46
freq_from_ftw = (ftw_selected * fs) / (2**FCW)

In [21]:
# ============================================================
# ===== PLOTLY DASHBOARD GENERATION =====
# ============================================================

fig = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=(
        "1. Real-time Extracted Target Amplitude (Calculated R Vector)",
        "2. Low-Pass Filtered Quadrature Components (DC Levels)",
        "3. Raw Phase Checkpoint Alignment (ADC vs Reference Sine)",
        "4. Loop Pulse Engine Edge Checkpoints (ADPLL Control Channels)",
        "5. Running Coarse Frame Estimator Timing Windows (Period)",
        "6. Final Resolved System Clock Frequency (Tracking Output)"
    )
)

# Row 1: Vector Amplitude Metrics
fig.add_trace(go.Scatter(x=t, y=calculated_amplitude, name="Calculated Amplitude (R)", line=dict(color="darkgreen", width=2.5)), row=1, col=1)

# Row 2: Quadrature components (I & Q)
fig.add_trace(go.Scatter(x=t, y=mix_i_lpf, name="In-Phase Filtered Line (I)", line=dict(color="blue")), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=mix_q_lpf, name="Quadrature Filtered Line (Q)", line=dict(color="purple")), row=2, col=1)

# Row 3: Physical Signal Phase Tracking
fig.add_trace(go.Scatter(x=t, y=adc_aligned, name="Raw Aligned Input (ADC)", line=dict(color="gray", width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=sin_ref, name="Internal NCO Sine Wave Reference", line=dict(color="crimson", dash="dash")), row=3, col=1)

# Row 4: ADPLL Edge Activity Pulses
fig.add_trace(go.Scatter(x=t_freq, y=ref_edge_pll, name="Ref PLL Input Edge Trigger", line=dict(color="orange")), row=4, col=1)
fig.add_trace(go.Scatter(x=t_freq, y=nco_edge_pll, name="NCO PLL Feedback Edge Trigger", line=dict(color="teal")), row=4, col=1)

# Row 5: Internal Calculated Period Base
fig.add_trace(go.Scatter(x=t_freq, y=period_avg, name="Calculated Average Period Frame Time", line=dict(color="chocolate")), row=5, col=1)

# Row 6: Frequency Outputs
fig.add_trace(go.Scatter(x=t_freq, y=freq_from_ftw, name="Absolute Frequency (Hz)", line=dict(color="black", width=2)), row=6, col=1)


# Global Layout Styling Adjustments
fig.update_layout(
    height=1500,
    title="Lock-In Amplifier & Extraction Core System Debug Performance Evaluation Dashboard",
    hovermode="x unified",
    showlegend=True
)

# Apply common axis configuration labels
for i in range(1, 7):
    fig.update_yaxes(title_text="Counts / Metrics", row=i, col=1)
fig.update_xaxes(title_text="Time (Seconds)", row=6, col=1)

fig.show()

In [22]:
import pandas as pd
from datetime import datetime

# ============================================================
# ===== EXPORT AMPLITUDE & TIME MATRIX TO CSV FILE =====
# ============================================================

# 1. Calculate correct multi-rate time indexing array
# 'decimation_rate' is what you sent to the FPGA (e.g., from your set_lpf_coefficients script)
# If your variable is named differently, update it here.
decimation_rate = 1
effective_fs = fs / decimation_rate  
t_sampled = np.arange(len(calculated_amplitude)) / effective_fs

# 2. Package the synchronized vector arrays into a structured Pandas DataFrame
df_export = pd.DataFrame({
    'Time (Seconds)': t_sampled,
    'Amplitude R (Volts)': calculated_amplitude
})

# 3. Create a clean, unique timestamped destination file string name
csv_filename = f"lockin_amplitude_trace_{datetime.now().strftime('%Y%m%d_%H%M%S')}_68.2873kHz_from_fg_1ms_time_constant_filter_not_averaged.csv"

# 4. Write data block down to disk (ignoring redundant index keys)
df_export.to_csv(csv_filename, index=False)

print("\n==============================================================")
print(f"EXPORT COMPLETE: Clear dataset written successfully.")
print(f" -> Destination File Name: {csv_filename}")
print(f" -> Total Rows Logged:     {len(df_export)} elements")
print(f" -> Sampling Delta (dt):   {1.0 / effective_fs * 1e6:.3f} µs")
print("==============================================================")


EXPORT COMPLETE: Clear dataset written successfully.
 -> Destination File Name: lockin_amplitude_trace_20260527_153305_68.2873kHz_from_fg_1ms_time_constant_filter_not_averaged.csv
 -> Total Rows Logged:     1000000 elements
 -> Sampling Delta (dt):   0.008 µs


In [23]:
import numpy as np
from scipy.signal import detrend
import plotly.graph_objects as go

# ============================================================
# ===== FFT CONFIGURATION & USER LIMITS =====
# ============================================================
effective_fs = fs  # Post-decimation sample rate
N = len(calculated_amplitude)

# Define your initial zoom window here (in Hz)
FREQ_MIN_HZ = 0.0       
FREQ_MAX_HZ = 500000.0   

# ============================================================
# ===== FFT COMPUTATION =====
# ============================================================
window = np.hanning(N)
windowed_signal = calculated_amplitude * window

REMOVE_DC = True 
if REMOVE_DC:
    fft_input = detrend(windowed_signal)
else:
    fft_input = windowed_signal

fft_raw = np.fft.rfft(fft_input)       
fft_freqs = np.fft.rfftfreq(N, d=1.0/effective_fs)

magnitude_volts = (np.abs(fft_raw) / N) * 2.0 * 2.0 
magnitude_dbv = 20 * np.log10(magnitude_volts + 1e-12)

# Convert arrays to kHz for easier reading on the plot
freqs_khz = fft_freqs / 1e3

# ============================================================
# ===== INTERACTIVE PLOTLY SPECTRUM =====
# ============================================================
fig = go.Figure()

# Add the FFT spectrum line
fig.add_trace(go.Scatter(
    x=freqs_khz, 
    y=magnitude_dbv,
    mode='lines',
    name='R Amplitude FFT',
    line=dict(color='crimson', width=1.5),
    hovertemplate='<b>Frequency:</b> %{x:.3f} kHz<br><b>Magnitude:</b> %{y:.2f} dBV<extra></extra>'
))

# Configure layout, titles, and structural zoom limits
y_label = "Magnitude (dBV RMS - AC Coupled)" if REMOVE_DC else "Magnitude (dBV RMS)"

fig.update_layout(
    title="FFT Spectrum of Lock-In Vector Amplitude (R)",
    xaxis_title="Frequency (kHz)",
    yaxis_title=y_label,
    xaxis=dict(range=[FREQ_MIN_HZ / 1e3, FREQ_MAX_HZ / 1e3]),  # Initial display limits
    hovermode="x unified",                                     # Clean hover tracking line
    template="plotly_white"
)

# Display the plot
fig.show()

In [24]:
import pandas as pd
from datetime import datetime

# ============================================================
# ===== EXPORT FFT FREQUENCY SPECTRUM TO CSV FILE =====
# ============================================================

# 1. Package the synchronized Frequency (kHz) and Magnitude (dBV) into a DataFrame
df_fft_export = pd.DataFrame({
    'Frequency (kHz)': freqs_khz,
    'Magnitude (dBV)': magnitude_dbv
})

# 2. Optional: If you only want to save the frequencies within your zoomed view
# Uncomment the line below to filter the CSV rows to your visible window bounds:
# df_fft_export = df_fft_export[(df_fft_export['Frequency (kHz)'] >= FREQ_MIN_HZ/1e3) & (df_fft_export['Frequency (kHz)'] <= FREQ_MAX_HZ/1e3)]

# 3. Create a clean, unique timestamped filename
fft_filename = f"lockin_fft_spectrum_{datetime.now().strftime('%Y%m%d_%H%M%S')}__68.2873kHz_from_fg_1ms_time_constant_filter_not_averaged.csv"

# 4. Write data block down to disk (ignoring redundant index keys)
df_fft_export.to_csv(fft_filename, index=False)

print("\n==============================================================")
print(f"FFT EXPORT COMPLETE: Spectrum dataset written successfully.")
print(f" -> Destination File Name: {fft_filename}")
print(f" -> Total Spectral Lines:  {len(df_fft_export)} bins")
print(f" -> Frequency Step (df):   {((fft_freqs[1] - fft_freqs[0])):.3f} Hz")
print("==============================================================")


FFT EXPORT COMPLETE: Spectrum dataset written successfully.
 -> Destination File Name: lockin_fft_spectrum_20260527_153307__68.2873kHz_from_fg_1ms_time_constant_filter_not_averaged.csv
 -> Total Spectral Lines:  500001 bins
 -> Frequency Step (df):   125.000 Hz
